Principal Component Analysis

Generated by MLguide. This notebook is a starter workflow for **PCA-based anomaly detection**.

# Problem context
_No problem description provided._

# 1. Environment setup

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib

# 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 3. Load and inspect data

In [ ]:
data_path = "data.csv"

# Optional: set this to your anomaly indicator column if available (0=normal, 1=anomaly).
label_column = None

# Quantile used to set anomaly threshold on reconstruction error.
anomaly_quantile = 0.99

# Keep enough components to explain this fraction of variance.
explained_variance_target = 0.95

df = pd.read_csv(data_path)
print(df.head())
print(f"Rows: {len(df)}, Columns: {len(df.columns)}")

# 4. Feature preprocessing

In [ ]:
if label_column and label_column in df.columns:
    feature_df = df.drop(columns=[label_column])
    y = df[label_column].astype(int)
else:
    feature_df = df.copy()
    y = None

numeric_cols = feature_df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in feature_df.columns if c not in numeric_cols]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_cols,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_cols,
        ),
    ],
    remainder="drop",
)

X_all = preprocess.fit_transform(feature_df)
if hasattr(X_all, "toarray"):
    X_all = X_all.toarray()

if y is not None and (y == 0).any():
    X_train = X_all[y == 0]
    print(f"Training PCA on normal samples only: {X_train.shape[0]} rows")
else:
    X_train = X_all
    print("No normal/anomaly labels provided. Training PCA on all rows.")

# 5. Fit PCA and score anomalies

In [ ]:
pca = PCA(n_components=explained_variance_target, svd_solver="full", random_state=42)
pca.fit(X_train)

X_proj = pca.transform(X_all)
X_recon = pca.inverse_transform(X_proj)

reconstruction_error = np.mean((X_all - X_recon) ** 2, axis=1)
threshold = float(np.quantile(reconstruction_error, anomaly_quantile))
pred_anomaly = (reconstruction_error >= threshold).astype(int)

print(f"Selected components: {pca.n_components_}")
print(f"Anomaly threshold (q={anomaly_quantile}): {threshold:.6f}")
print(f"Predicted anomalies: {int(pred_anomaly.sum())}/{len(pred_anomaly)}")

# 6. Evaluate (if labels are available)

In [ ]:
if y is not None:
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(y, pred_anomaly))
    print("\nClassification report:")
    print(classification_report(y, pred_anomaly, digits=4))
else:
    print("No label column configured; skipping supervised evaluation.")

# 7. Inspect anomaly scores

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(reconstruction_error, linewidth=1)
plt.axhline(threshold, color="red", linestyle="--", label=f"threshold={threshold:.4f}")
plt.title("PCA reconstruction error")
plt.xlabel("Sample index")
plt.ylabel("Reconstruction error")
plt.legend()
plt.tight_layout()
plt.show()

# 8. Save scored output

In [ ]:
output = df.copy()
output["anomaly_score"] = reconstruction_error
output["predicted_anomaly"] = pred_anomaly

output_path = "pca_anomaly_scores.csv"
output.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

# 9. Method notes
- PCA anomaly detection assumes normal data lies in a lower-dimensional subspace.
- Tune `explained_variance_target` and `anomaly_quantile` for your precision/recall tradeoff.
- For severe class imbalance, evaluate with precision-recall curves and domain-specific cost metrics.